以下是代码的逐行详细解释，用于绘制逻辑回归的决策边界和分类结果：

---

### **1. 创建图形窗口**
```python
fig, ax = plt.subplots(figsize=(12, 8))
```
- **作用**：创建一个大小为 `12×8` 英寸的图形窗口。
- **变量**：
  - `fig`：整个图形对象。
  - `ax`：子图对象，用于后续绘图操作。

---

### **2. 绘制正负样本点**
```python
ax.scatter(positive['x'], positive['y'], s=50, c='b', marker='o', label='Will Pay')
ax.scatter(negative['x'], negative['y'], s=50, c='r', marker='x', label='Will Not Pay')
```
- **作用**：在图上绘制两类样本点。
  - **正类**（`Outcome=1`）：蓝色圆点 (`o`)，标记为 "Will Pay"。
  - **负类**（`Outcome=0`）：红色叉号 (`x`)，标记为 "Will Not Pay"。
- **参数**：
  - `s=50`：点的大小。
  - `label`：用于图例的标签。

---

### **3. 生成网格坐标**
```python
x = np.linspace(data2['x'].min(), data2['x'].max(), 100)
y = np.linspace(data2['y'].min(), data2['y'].max(), 100)
X_grid, Y_grid = np.meshgrid(x, y)
```
- **作用**：创建覆盖数据范围的网格点，用于计算决策边界。
  - `np.linspace`：在 `x` 和 `y` 的最小/最大值之间生成 100 个等间距点。
  - `np.meshgrid`：将 `x` 和 `y` 转换为网格坐标矩阵 `X_grid` 和 `Y_grid`（均为 100×100 的数组）。

---

### **4. 计算决策边界概率**
```python
Z = np.zeros(X_grid.shape)
for i in range(X_grid.shape[0]):
    for j in range(X_grid.shape[1]):
        features = np.array([1, X_grid[i, j], Y_grid[i, j]])
        for k in range(2, degree + 1):
            for l in range(0, k + 1):
                features = np.append(features, (X_grid[i, j] ** (k - l)) * (Y_grid[i, j] ** l))
        Z[i, j] = sigmoid(np.dot(features, theta))
```
- **作用**：对每个网格点计算逻辑回归的预测概率。
  - **步骤**：
    1. 初始化 `Z` 为与网格相同形状的零数组（100×100）。
    2. 遍历每个网格点 `(X_grid[i,j], Y_grid[i,j])`：
       - 构造特征向量 `features`，包含偏置项 `1`、原始特征 `x` 和 `y`，以及所有多项式组合（如 `x²`, `xy`, `y²`, ..., 直到 `degree=5`）。
       - 计算该点的预测概率：`sigmoid(features @ theta)`，结果存入 `Z[i,j]`。
  - **注意**：这里使用了之前生成的模型参数 `theta` 和多项式最高次数 `degree`。

---

### **5. 绘制决策边界**
```python
ax.contour(X_grid, Y_grid, Z, levels=[0.5], colors='g', linewidths=2, linestyles='dashed', label='Decision Boundary')
```
- **作用**：在概率值 `Z=0.5` 处绘制决策边界。
  - **参数**：
    - `levels=[0.5]`：指定等高线的阈值（逻辑回归中 `0.5` 是分类阈值）。
    - `colors='g'`：绿色边界线。
    - `linestyles='dashed'`：虚线样式。
    - `label`：图例标签。

---

### **6. 添加标签和图例**
```python
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Logistic Regression Decision Boundary with Regularization')
ax.legend()
plt.show()
```
- **作用**：完善图形显示。
  - `set_xlabel/set_ylabel`：设置坐标轴标签。
  - `set_title`：设置标题（注明模型和正则化）。
  - `legend()`：显示图例。
  - `plt.show()`：渲染图形。

---

### **关键点总结**
1. **决策边界原理**：通过网格点计算模型预测概率，在 `p=0.5` 处绘制等高线。
2. **多项式特征**：代码中动态生成了所有可能的 `x` 和 `y` 的组合（如 `x²y`, `xy³` 等），与训练时的特征工程一致。
3. **性能注意**：双重循环计算 `Z` 效率较低，实际应用中可用向量化优化（如 `np.polyval` 或提前生成多项式特征矩阵）。

---

### **改进建议**
```python
# 向量化计算示例（需提前定义多项式特征生成函数）
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=degree, include_bias=True)
grid_points = np.column_stack([X_grid.ravel(), Y_grid.ravel()])
grid_features = poly.fit_transform(grid_points)
Z = sigmoid(grid_features @ theta).reshape(X_grid.shape)
```